# 42ndAlignment Colab Notebook

Clean runnable notebook for training and evaluating the Epistemic Maturity LoRA adapters on `gpt-oss-20b`.

This notebook replaces the messy working session. It is intentionally linear:

1. Setup environment
2. Load `gpt-oss-20b` through Unsloth
3. Train v0.1 SFT LoRA
4. Save/download v0.1
5. Optional: train v0.2 preference-derived SFT
6. Optional: run streaming evaluation

Use **Runtime → Change runtime type → T4 GPU** before running.

## 0. Runtime Check

Run this first. It confirms whether Colab gave you a GPU.

In [ ]:
!nvidia-smi

## 1. Install Dependencies

Run this cell. After it completes, run the Hugging Face Hub repair cell below, then restart the session once.

Reason: the default Colab/Unsloth environment can produce a `huggingface_hub` mismatch.

In [ ]:
%%capture
import os, importlib.util

!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"
        _pil = "pillow"

    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

## 2. Hugging Face Hub Repair Cell

Run this after the install cell. Then choose **Runtime → Restart session**.

After restart, run the install cell and this repair cell again before continuing.

In [ ]:
!pip uninstall -y huggingface_hub hf_transfer
!pip install -q --force-reinstall "huggingface_hub==0.36.0" hf_transfer

## 3. Optional Hugging Face Login

Only needed if model access asks for a token.

Use a read-only token. Do not paste tokens into notebook cells.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Upload Dataset Files

For v0.1 SFT training, upload:

```text
epistemic_maturity_sft_chat_v1_300.jsonl
```

For v0.2 preference-derived SFT, also upload:

```text
epistemic_maturity_dpo_pairs_v1_300.jsonl
```

For evaluation, also upload:

```text
epistemic_maturity_eval_prompts_v1.jsonl
```

In [ ]:
from google.colab import files
uploaded = files.upload()
list(uploaded.keys())

## 5. Load Base Model

T4-safe settings:

```text
model_name = unsloth/gpt-oss-20b-unsloth-bnb-4bit
max_seq_length = 512
load_in_4bit = True
```

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",
    dtype = dtype,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    full_finetuning = False,
)

## 6. Add LoRA Adapter

This creates trainable LoRA layers. It does not train the full model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# Train v0.1 SFT Adapter

This stage trains on the 300 curated Epistemic Maturity SFT examples.

In [ ]:
from datasets import load_dataset

sft_dataset = load_dataset(
    "json",
    data_files = "epistemic_maturity_sft_chat_v1_300.jsonl",
    split = "train",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False,
            reasoning_effort = "medium",
        )
        for convo in convos
    ]
    return { "text" : texts }

sft_dataset = sft_dataset.map(formatting_prompts_func, batched = True)
sft_dataset = sft_dataset.remove_columns([col for col in sft_dataset.column_names if col != "text"])

sft_dataset

## 7. Configure SFT Trainer

This first proof run uses 60 steps.

If the run is slow, reduce `max_steps` to 30. If the run is stable and you have quota, increase it later.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = sft_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_sft",
        report_to = "none",
    ),
)

## 8. Train v0.1

If response-only masking causes `num_samples=0`, skip masking and run this cell directly. This clean notebook intentionally avoids response-only masking for reliability on Colab T4.

In [ ]:
sft_stats = sft_trainer.train()

## 9. Save and Download v0.1 Adapter

Run this immediately after training finishes.

In [ ]:
model.save_pretrained("epistemic_gpt_oss_20b_sft_lora_v0_1")
tokenizer.save_pretrained("epistemic_gpt_oss_20b_sft_lora_v0_1")

!zip -r epistemic_gpt_oss_20b_sft_lora_v0_1.zip epistemic_gpt_oss_20b_sft_lora_v0_1

from google.colab import files
files.download("epistemic_gpt_oss_20b_sft_lora_v0_1.zip")

# Optional: Train v0.2 Preference-Derived SFT

This is the fallback used after true DPO failed on the free Colab T4 stack with a dtype mismatch.

This is **not true DPO**. It trains on the `chosen` answers from the DPO-style preference dataset.

## 10. Load Preference Dataset

Run this only if you uploaded:

```text
epistemic_maturity_dpo_pairs_v1_300.jsonl
```

In [ ]:
from datasets import load_dataset

pref_dataset = load_dataset(
    "json",
    data_files = "epistemic_maturity_dpo_pairs_v1_300.jsonl",
    split = "train",
)

pref_dataset

## 11. Convert Chosen Responses to SFT Text

In [ ]:
SYSTEM_PROMPT = "Answer with epistemic maturity: track reality, handle tradeoffs, avoid strawmen, calibrate certainty, and preserve the user's agency."

def dpo_chosen_to_sft(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["chosen"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize = False,
        add_generation_prompt = False,
        reasoning_effort = "medium",
    )

    return {"text": text}

pref_sft_dataset = pref_dataset.map(
    dpo_chosen_to_sft,
    remove_columns = pref_dataset.column_names,
)

pref_sft_dataset[0]

## 12. Train v0.2 Preference-Derived SFT

In [ ]:
from trl import SFTConfig, SFTTrainer

pref_sft_trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = pref_sft_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 30,
        learning_rate = 1e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_pref_sft",
        report_to = "none",
    ),
)

pref_sft_stats = pref_sft_trainer.train()

## 13. Save and Download v0.2 Adapter

In [ ]:
model.save_pretrained("epistemic_gpt_oss_20b_sft_pref_sft_lora_v0_2")
tokenizer.save_pretrained("epistemic_gpt_oss_20b_sft_pref_sft_lora_v0_2")

!zip -r epistemic_gpt_oss_20b_sft_pref_sft_lora_v0_2.zip epistemic_gpt_oss_20b_sft_pref_sft_lora_v0_2

from google.colab import files
files.download("epistemic_gpt_oss_20b_sft_pref_sft_lora_v0_2.zip")

# Optional: Streaming Evaluation

Use this section to generate outputs while saving each prompt immediately.

This avoids losing all progress if Colab disconnects.

In [ ]:
import json

eval_prompts = []
with open("epistemic_maturity_eval_prompts_v1.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        eval_prompts.append(json.loads(line))

len(eval_prompts), eval_prompts[0]["id"]

In [ ]:
import json, torch, os

def generate_eval_outputs_streaming(model, tokenizer, eval_prompts, output_path, max_new_tokens=384):
    completed_ids = set()

    if os.path.exists(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    completed_ids.add(json.loads(line)["id"])

    with open(output_path, "a", encoding="utf-8") as f:
        for item in eval_prompts:
            if item["id"] in completed_ids:
                print(f"skip {item['id']}")
                continue

            messages = [
                {
                    "role": "system",
                    "content": "Answer with epistemic maturity: track reality, handle tradeoffs, avoid strawmen, calibrate certainty, and preserve the user's agency."
                },
                {
                    "role": "user",
                    "content": item["prompt"]
                },
            ]

            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt = True,
                return_tensors = "pt",
                return_dict = True,
                reasoning_effort = "medium",
            ).to("cuda")

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens = max_new_tokens,
                    do_sample = False,
                    temperature = None,
                    top_p = None,
                )

            text = tokenizer.decode(outputs[0], skip_special_tokens=False)

            row = {
                "id": item["id"],
                "category": item["category"],
                "prompt": item["prompt"],
                "response": text,
                "primary_gates": item.get("primary_gates", []),
                "target_collapse_flags": item.get("target_collapse_flags", []),
                "eval_focus": item.get("eval_focus", "")
            }

            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()

            print(f"saved {item['id']}")

## 14. Generate First 20 Evaluation Outputs

Run this for the currently loaded model.

For base model evaluation, use a fresh runtime and load only the base model before running this section.
For adapter evaluation, load the adapter before running this section.

In [ ]:
rows = generate_eval_outputs_streaming(
    model,
    tokenizer,
    eval_prompts[:20],
    "eval_outputs_20_384.jsonl",
    max_new_tokens = 384,
)

from google.colab import files
files.download("eval_outputs_20_384.jsonl")

# Notes

## True DPO Status

True DPO was attempted during development, but the free Colab T4 `gpt-oss + Unsloth + TRL` stack failed with a dtype mismatch:

```text
RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != c10::Half
```

The v0.2 adapter uses preference-derived SFT as a practical fallback. It should not be described as DPO.

## Tokenizer Note

If `tokenizer.json` is too large for GitHub upload, omit it from the repo. The tokenizer can be loaded from the base model.